# Hey Sense Model Layer + Need Detection + Next Best Action

## Fase 3: Model Layer, Need Detection Engine y Next Best Action Engine

Este notebook consume los outputs de Fase 2 y construye la capa inteligente de Hey Sense.

Contrato del pipeline:

```text
Fase 1 -> data_processed/
Fase 2 -> data_features/customer_360.pkl
Fase 3 -> data_model_outputs/
```

Objetivos de esta fase:

- ejecutar modelos no supervisados para segmentacion;
- detectar anomalias y riesgo operativo;
- enriquecer scores de propension y riesgo;
- inferir necesidades implicitas por cliente;
- seleccionar la siguiente mejor accion;
- generar salidas listas para dashboard, demo y pitch.

## 1. Imports y carga de Customer 360

Este notebook no repite limpieza ni feature engineering. Solo consume `customer_360.pkl` generado por Fase 2.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from textwrap import fill

from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)

BASE_DIR = Path.cwd()
FEATURE_DIR = BASE_DIR / "data_features"
MODEL_DIR = BASE_DIR / "data_model_outputs"
MODEL_DIR.mkdir(exist_ok=True)

CUSTOMER_360_PATH = FEATURE_DIR / "customer_360.pkl"
if not CUSTOMER_360_PATH.exists():
    raise FileNotFoundError("Falta data_features/customer_360.pkl. Ejecuta primero la Fase 2.")

customer_360 = pd.read_pickle(CUSTOMER_360_PATH)
print("Customer 360 cargado:", customer_360.shape)
display(customer_360.head())

Customer 360 cargado: (15025, 107)


,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico,satisfaccion_1_10_was_missing,estado_was_missing,ciudad_was_missing,total_interacciones_havi,total_conversaciones,total_mensajes_voz,total_mensajes_texto,canal_preferido_havi,sentimiento_promedio,intent_cargo_no_reconocido,intent_consulta_movimiento,intent_credito,intent_promocion,intent_cashback,intent_bloqueo_tarjeta,intent_cancelacion,intent_soporte_app,intent_asesor_humano,ratio_voz_texto,mensajes_por_conversacion,repeticion_problema_score,frustracion_score,urgencia_score,intencion_dominante,temas_recurrentes,total_transacciones,monto_total,monto_promedio,monto_mediano,monto_max,num_no_procesadas,num_disputas,num_revertidas,num_reintentos,num_transacciones_internacionales,num_patrones_atipicos,rechazos_saldo_insuficiente,rechazos_limite_excedido,tx_fin_semana,tx_nocturnas,cashback_total,meses_diferidos_promedio,porcentaje_gasto_fin_semana,porcentaje_gasto_nocturno,ratio_no_procesadas,ratio_disputas,ratio_atipicas,ratio_internacional,gasto_educacion,gasto_entretenimiento,gasto_gobierno,gasto_hogar,gasto_restaurante,gasto_retiro_cajero,gasto_ropa_accesorios,gasto_salud,gasto_servicios_digitales,gasto_supermercado,gasto_tecnologia,gasto_transferencia,gasto_transporte,gasto_viajes,gasto_delivery,tiene_tarjeta_credito,tiene_inversion,tiene_seguro_producto,tiene_credito_personal,tiene_credito_auto,num_productos,num_productos_activos_calc,utilizacion_credito_max,utilizacion_credito_promedio,limite_credito_total,saldo_total,mensualidad_total_creditos,tasa_interes_promedio,liquidez_score,estres_financiero_score,capacidad_ahorro_score,riesgo_friccion_score,riesgo_fraude_score,propension_credito_score,propension_inversion_score,propension_seguro_score,churn_risk_score
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0,False,True,es_MX,False,2,False,False,False,False,3,1,0,3,texto,0.0,0,1,2,0,0,2,0,0,0,0.0,3.0,11.1,0.0,0.0,credito,"credito, bloqueo_tarjeta, consulta_movimiento",56,119570.59,2135.189107,437.370,24010.0,3,0,0,2,2,0,1,0,20,15,122.33,0.000000,0.357143,0.267857,0.053571,0.000000,0.0,0.035714,0.0,3038.55,2057.95,0.0,6040.89,0.0,665.36,0.00,6671.82,839.38,0.00,100256.64,0.0,0.00,0,1,0,0,0,0,2,2,0.6166,0.308300,144000.0,169745.00,0.0,17.855,32.84,21.44,33.59,4.29,3.08,28.68,64.75,52.02,4.44
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,False,App,714,3,app_android,8.0,False,True,es_MX,True,2,False,False,False,False,3,1,0,3,texto,0.0,0,0,2,0,0,0,0,0,1,0.0,3.0,11.1,0.0,8.33,credito,"credito, asesor_humano",74,212722.67,2874.630676,425.615,24830.0,1,1,1,1,1,0,1,0,15,13,147.08,0.000000,0.202703,0.175676,0.013514,0.013514,0.0,0.013514,0.0,2344.75,0.00,0.0,8058.26,0.0,2215.13,0.00,6411.86,1400.74,409.21,191882.72,0.0,0.00,0,1,0,0,0,0,2,2,0.2783,0.139150,22000.0,26835.14,0.0,17.040,25.60,10.82,20.62,11.21,4.66,35.94,55.28,16.21,10.67
2,USR-00003,23,H,Chihuahua,Cuauhtémoc,Licenciatura,Estudiante,14000,1174,True,False,App,454,3,app_ios,8.0,False,True,es_MX,False,2,False,False,False,False,3,1,0,3,texto,0.0,0,0,0,0,0,0,0,0,0,0.0,3.0,11.1,0.0,0.0,sin_intencion_detectada,sin_tema_detectado,79,190801.09,2415.203671,559.020,24540.0,4,3,1,1,4,0,2,1,18,17,165.43,0.000000,0.227848,0.215190,0.050633,0.037975,0.0,0.050633,0.0,2872.38,10322.28,0.0,4682.51,0.0,2802.62,0.00,9028.92,554.59,1620.92,158916.87,0.0,0.00,0,1,0,0,0,0,2,2,0.3091,0.154550,5000.0,5000.15,0.0,19.940,16.77,19.37,17.98,17.5,15.53,19.17,52.98,48.13,14.39
3,USR-00004,32,SE,Nuevo León,Guadalupe,Posgrado,Empleado,61000,1168,False,False,Fan Shop,837,16,app_ios,10.0,True,False,es_MX,True,3,False,False,False,False,3,1,0,3,texto,-0.666667,0,0,2,0,0,2,2,0,0,0.0,3.0,11.1,22.22

## 1.1 Utilidades de scoring

Funciones auxiliares para escalar scores de modelos y reglas de decision.


In [2]:
def minmax_score(series, higher_is_better=True):
    s = pd.to_numeric(series, errors="coerce").fillna(0)
    min_v = s.min()
    max_v = s.max()
    if max_v == min_v:
        score = pd.Series(50, index=s.index, dtype="float64")
    else:
        score = 100 * (s - min_v) / (max_v - min_v)
    if not higher_is_better:
        score = 100 - score
    return score.clip(0, 100).round(2)


## 2. Seleccion de variables para modelado

Se separan variables numericas accionables para modelos no supervisados.

Criterios:

- usar comportamiento financiero, conversacional y de producto;
- evitar identificadores y texto libre;
- no usar variables que sean salida final de esta fase;
- conservar interpretabilidad para explicar segmentos y acciones.

In [3]:
model_feature_candidates = [
    # perfil cliente
    "edad", "ingreso_mensual_mxn", "antiguedad_dias", "score_buro", "dias_desde_ultimo_login", "satisfaccion_1_10",
    "es_hey_pro", "nomina_domiciliada", "recibe_remesas", "usa_hey_shop", "tiene_seguro", "num_productos_activos", "patron_uso_atipico",
    # conversacional
    "total_interacciones_havi", "total_conversaciones", "ratio_voz_texto", "mensajes_por_conversacion",
    "frustracion_score", "urgencia_score", "repeticion_problema_score",
    "intent_cargo_no_reconocido", "intent_consulta_movimiento", "intent_credito", "intent_promocion", "intent_cashback",
    "intent_bloqueo_tarjeta", "intent_cancelacion", "intent_soporte_app", "intent_asesor_humano",
    # transaccional
    "total_transacciones", "monto_total", "monto_promedio", "monto_mediano", "monto_max",
    "num_no_procesadas", "num_disputas", "num_revertidas", "num_reintentos", "num_transacciones_internacionales", "num_patrones_atipicos",
    "rechazos_saldo_insuficiente", "rechazos_limite_excedido", "porcentaje_gasto_fin_semana", "porcentaje_gasto_nocturno",
    "ratio_no_procesadas", "ratio_disputas", "ratio_atipicas", "ratio_internacional",
    "gasto_delivery", "gasto_restaurante", "gasto_supermercado", "gasto_servicios_digitales", "gasto_entretenimiento", "gasto_tecnologia",
    # producto
    "tiene_tarjeta_credito", "tiene_inversion", "tiene_seguro_producto", "tiene_credito_personal", "tiene_credito_auto",
    "num_productos", "num_productos_activos_calc", "utilizacion_credito_max", "utilizacion_credito_promedio",
    "limite_credito_total", "saldo_total", "mensualidad_total_creditos", "tasa_interes_promedio",
    # scores fase 2
    "liquidez_score", "estres_financiero_score", "capacidad_ahorro_score", "riesgo_friccion_score", "riesgo_fraude_score",
    "propension_credito_score", "propension_inversion_score", "propension_seguro_score", "churn_risk_score",
]

model_features = [col for col in model_feature_candidates if col in customer_360.columns]
X_model = customer_360[model_features].copy()

# Convertir booleanos nullable a enteros modelables.
for col in X_model.columns:
    if str(X_model[col].dtype) in ["boolean", "bool"]:
        X_model[col] = X_model[col].astype(int)

X_model = X_model.apply(pd.to_numeric, errors="coerce").fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_model)

print("Features usadas para modelos:", len(model_features))
display(pd.DataFrame({"feature": model_features}).head(80))

Features usadas para modelos: 76


,feature
0,edad
1,ingreso_mensual_mxn
2,antiguedad_dias
3,score_buro
4,dias_desde_ultimo_login
5,satisfaccion_1_10
6,es_hey_pro
7,nomina_domiciliada
8,recibe_remesas
9,usa_hey_shop


## 3. NLP para Havi: intenciones, friccion y resumen conversacional

La clasificacion de intenciones se hereda de Fase 2 porque ya fue extraida desde conversaciones con reglas NLP.

Aqui se construyen salidas de negocio por cliente:

- `intencion_dominante`;
- `temas_recurrentes`;
- `conversation_summary`;
- nivel de friccion conversacional.

Esto funciona como baseline explicable. En produccion se puede reemplazar por un clasificador supervisado o embeddings.

In [4]:
def friction_level(score):
    if score >= 70:
        return "alta"
    if score >= 40:
        return "media"
    return "baja"


def build_conversation_summary(row):
    intent = row.get("intencion_dominante", "sin_intencion_detectada")
    temas = row.get("temas_recurrentes", "sin_tema_detectado")
    friccion = friction_level(row.get("frustracion_score", 0))
    urgencia = friction_level(row.get("urgencia_score", 0))
    interacciones = int(row.get("total_interacciones_havi", 0))
    if interacciones == 0:
        return "Cliente sin interacciones recientes con Havi."
    return f"Cliente con {interacciones} interacciones con Havi. Intencion dominante: {intent}. Temas: {temas}. Friccion {friccion}, urgencia {urgencia}."

customer_360["friccion_conversacional_nivel"] = customer_360["frustracion_score"].apply(friction_level)
customer_360["urgencia_conversacional_nivel"] = customer_360["urgencia_score"].apply(friction_level)
customer_360["conversation_summary"] = customer_360.apply(build_conversation_summary, axis=1)

customer_360[["user_id", "intencion_dominante", "temas_recurrentes", "friccion_conversacional_nivel", "urgencia_conversacional_nivel", "conversation_summary"]].head(10)

,user_id,intencion_dominante,temas_recurrentes,friccion_conversacional_nivel,urgencia_conversacional_nivel,conversation_summary
0,USR-00001,credito,"credito, bloqueo_tarjeta, consulta_movimiento",baja,baja,Cliente con 3 interacciones con Havi. Intencio...
1,USR-00002,credito,"credito, asesor_humano",baja,baja,Cliente con 3 interacciones con Havi. Intencio...
2,USR-00003,sin_intencion_detectada,sin_tema_detectado,baja,baja,Cliente con 3 interacciones con Havi. Intencio...
3,USR-00004,credito,"credito, bloqueo_tarjeta, cancelacion",baja,baja,Cliente con 3 interacciones con Havi. Intencio...
4,USR-00005,credito,credito,baja,baja,Cliente con 5 interacciones con Havi. Intencio...
5,USR-00006,consulta_movimiento,consulta_movimiento,baja,baja,Cliente con 1 interacciones con Havi. Intencio...
6,USR-00007,consulta_movimiento,consulta_movimiento,baja,baja,Cliente con 1 interacciones con Havi. Intencio...
7,USR-00008,bloqueo_tarjeta,"bloqueo_tarjeta, cancelacion",baja,baja,Cliente con 2 interacciones con Havi. Intencio...
8,USR-00009,bloqueo_tarjeta,bloqueo_tarjeta,baja,baja,Cliente con 3 interacciones con Havi. Intencio...
9,USR-00010,consulta_movimiento,"consulta_movimiento, bloqueo_tarjeta, cancelacion",baja,baja,Cliente con 2 interacciones con Havi. Intencio...


## 4. Clustering de clientes

Objetivo: descubrir segmentos automaticamente con KMeans.

Se evalua un rango de clusters y se elige una configuracion balanceada con:

- `silhouette_score`: separacion entre clusters;
- `davies_bouldin_score`: compacidad y separacion;
- interpretabilidad de segmentos.

En hackathon, la interpretabilidad pesa mucho: un cluster util debe convertirse en una accion de negocio.

In [5]:
cluster_metrics = []
for k in range(4, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    cluster_metrics.append({
        "k": k,
        "silhouette_score": silhouette_score(X_scaled, labels, sample_size=min(300, len(labels)), random_state=42),
        "davies_bouldin_score": davies_bouldin_score(X_scaled, labels),
    })

cluster_metrics_df = pd.DataFrame(cluster_metrics).sort_values("silhouette_score", ascending=False)
display(cluster_metrics_df)

# Se usa k=6 porque es la mejor configuracion segun silhouette y mantiene segmentos accionables.
K_SEGMENTS = 6
kmeans = KMeans(n_clusters=K_SEGMENTS, random_state=42, n_init=20)
customer_360["cluster_id"] = kmeans.fit_predict(X_scaled)

cluster_sizes = customer_360["cluster_id"].value_counts().sort_index().reset_index()
cluster_sizes.columns = ["cluster_id", "clientes"]
cluster_sizes



,k,silhouette_score,davies_bouldin_score
2,6,0.122376,2.175155
1,5,0.116462,2.336916
0,4,0.100142,2.492917
4,8,0.098702,2.305582
3,7,0.097332,2.310942


,cluster_id,clientes
0,0,3687
1,1,2051
2,2,1373
3,3,5436
4,4,1834
5,5,644


## 5. Interpretacion de segmentos

Se etiquetan clusters con reglas de interpretabilidad sobre los promedios de cada grupo.

Esto evita presentar clusters anonimos como `cluster 0`; cada segmento queda traducido a una narrativa de negocio.

In [6]:
segment_profile_cols = [
    "edad", "ingreso_mensual_mxn", "satisfaccion_1_10", "dias_desde_ultimo_login",
    "total_interacciones_havi", "total_transacciones", "monto_total",
    "num_no_procesadas", "num_disputas", "num_patrones_atipicos",
    "tiene_inversion", "tiene_tarjeta_credito", "tiene_seguro",
    "liquidez_score", "estres_financiero_score", "capacidad_ahorro_score", "riesgo_friccion_score", "riesgo_fraude_score",
    "propension_credito_score", "propension_inversion_score", "churn_risk_score",
]
segment_profile_cols = [c for c in segment_profile_cols if c in customer_360.columns]

segment_profiles = customer_360.groupby("cluster_id")[segment_profile_cols].mean().round(2)

def assign_unique_segment_labels(profiles):
    label_rules = [
        ("riesgo_fraude_score", "Alertas criticas de fraude"),
        ("estres_financiero_score", "Clientes con estres financiero"),
        ("churn_risk_score", "Clientes de bajo engagement"),
        ("propension_inversion_score", "Inversionistas de alto valor en vigilancia"),
        ("propension_credito_score", "Candidatos a credito"),
        ("capacidad_ahorro_score", "Ahorradores potenciales"),
        ("total_interacciones_havi", "Clientes digitales intensivos"),
        ("satisfaccion_1_10", "Clientes satisfechos consolidados"),
    ]
    fallback_labels = [
        "Clientes base transaccionales",
        "Clientes cotidianos de oportunidad",
        "Clientes mixtos por desarrollar",
    ]

    labels = {}
    remaining_clusters = set(profiles.index)

    for metric, label in label_rules:
        if not remaining_clusters or metric not in profiles.columns:
            continue
        cluster_id = profiles.loc[list(remaining_clusters), metric].idxmax()
        labels[cluster_id] = label
        remaining_clusters.remove(cluster_id)

    for cluster_id, label in zip(sorted(remaining_clusters), fallback_labels):
        labels[cluster_id] = label

    return labels

segment_labels = assign_unique_segment_labels(segment_profiles)
customer_360["segmento"] = customer_360["cluster_id"].map(segment_labels)

segment_summary = customer_360.groupby(["cluster_id", "segmento"]).agg(
    clientes=("user_id", "count"),
    ingreso_promedio=("ingreso_mensual_mxn", "mean"),
    satisfaccion_promedio=("satisfaccion_1_10", "mean"),
    fraude_promedio=("riesgo_fraude_score", "mean"),
    friccion_promedio=("riesgo_friccion_score", "mean"),
    inversion_promedio=("propension_inversion_score", "mean"),
    churn_promedio=("churn_risk_score", "mean"),
).round(2).reset_index()

segment_summary

,cluster_id,segmento,clientes,ingreso_promedio,satisfaccion_promedio,fraude_promedio,friccion_promedio,inversion_promedio,churn_promedio
0,0,Candidatos a credito,3687,43539.19,8.45,9.01,10.14,41.28,10.01
1,1,Clientes de bajo engagement,2051,19194.78,5.50,2.36,12.62,42.58,27.89
2,2,Inversionistas de alto valor en vigilancia,1373,59785.51,7.94,10.95,12.39,62.45,12.22
3,3,Ahorradores potenciales,5436,21565.77,8.22,9.46,11.01,54.83,10.76
4,4,Clientes con estres financiero,1834,15462.92,5.10,7.00,20.4,43.26,24.6
5,5,Alertas criticas de fraude,644,31174.69,7.88,30.39,11.96,50.57,12.33


## 5.1 Visualizacion de clusters

Se proyectan los clientes a dos componentes principales para visualizar la separacion de los 6 clusters. Se complementa con tamano de segmento y un mapa de calor de perfiles promedio para explicar la accion de negocio de cada grupo.

In [7]:
cluster_pca = pd.DataFrame(
    PCA(n_components=2).fit_transform(X_scaled),
    columns=["pca_1", "pca_2"],
    index=customer_360.index,
)
cluster_pca["cluster_id"] = customer_360["cluster_id"].values
cluster_pca["segmento"] = customer_360["segmento"].values
cluster_pca["user_id"] = customer_360["user_id"].values
cluster_pca["ingreso_mensual_mxn"] = customer_360["ingreso_mensual_mxn"].values
cluster_pca["satisfaccion_1_10"] = customer_360["satisfaccion_1_10"].values
cluster_pca["riesgo_fraude_score"] = customer_360["riesgo_fraude_score"].values
cluster_pca["churn_risk_score"] = customer_360["churn_risk_score"].values

cluster_ids = sorted(cluster_pca["cluster_id"].unique())
palette = ["#2563EB", "#DC2626", "#16A34A", "#F59E0B", "#7C3AED", "#0891B2", "#DB2777", "#4B5563"]
cluster_colors = {cluster_id: palette[i % len(palette)] for i, cluster_id in enumerate(cluster_ids)}

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"type": "scattergl"}, {"type": "bar"}], [{"type": "heatmap", "colspan": 2}, None]],
    subplot_titles=("Mapa PCA de clusters de clientes", "Tamano de cada segmento", "Perfil relativo por segmento"),
    row_heights=[0.58, 0.42],
    horizontal_spacing=0.13,
    vertical_spacing=0.18,
)

for cluster_id in cluster_ids:
    data = cluster_pca[cluster_pca["cluster_id"] == cluster_id]
    label = segment_labels.get(cluster_id, f"Cluster {cluster_id}")
    fig.add_trace(
        go.Scattergl(
            x=data["pca_1"],
            y=data["pca_2"],
            mode="markers",
            name=f"{cluster_id} - {label}",
            marker=dict(size=6, color=cluster_colors[cluster_id], opacity=0.62),
            customdata=data[["user_id", "segmento", "ingreso_mensual_mxn", "satisfaccion_1_10", "riesgo_fraude_score", "churn_risk_score"]],
            hovertemplate=(
                "Cliente: %{customdata[0]}<br>"
                "Segmento: %{customdata[1]}<br>"
                "Ingreso: $%{customdata[2]:,.0f}<br>"
                "Satisfaccion: %{customdata[3]:.1f}<br>"
                "Riesgo fraude: %{customdata[4]:.1f}<br>"
                "Churn: %{customdata[5]:.1f}<extra></extra>"
            ),
        ),
        row=1, col=1,
    )

centroids_2d = cluster_pca.groupby("cluster_id")[["pca_1", "pca_2"]].mean().reset_index()
fig.add_trace(
    go.Scatter(
        x=centroids_2d["pca_1"],
        y=centroids_2d["pca_2"],
        mode="markers+text",
        text=centroids_2d["cluster_id"],
        textposition="middle center",
        showlegend=False,
        marker=dict(size=24, color="white", line=dict(color="black", width=2)),
        hovertemplate="Centroide cluster %{text}<extra></extra>",
    ),
    row=1, col=1,
)

segment_counts = segment_summary.sort_values("clientes", ascending=True).copy()
fig.add_trace(
    go.Bar(
        x=segment_counts["clientes"],
        y=[fill(s, 30) for s in segment_counts["segmento"]],
        orientation="h",
        marker_color=[cluster_colors[c] for c in segment_counts["cluster_id"]],
        text=segment_counts["clientes"].map(lambda x: f"{x:,}"),
        textposition="outside",
        hovertemplate="%{y}<br>Clientes: %{x:,}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=2,
)

heatmap_cols = ["ingreso_promedio", "satisfaccion_promedio", "fraude_promedio", "friccion_promedio", "inversion_promedio", "churn_promedio"]
heatmap_data = segment_summary.set_index("segmento")[heatmap_cols].copy()
heatmap_data = heatmap_data.astype(float)
heatmap_scaled = (heatmap_data - heatmap_data.mean()) / heatmap_data.std(ddof=0).replace(0, 1)
fig.add_trace(
    go.Heatmap(
        z=heatmap_scaled.values,
        x=[c.replace("_promedio", "").replace("_", " ") for c in heatmap_cols],
        y=[fill(s, 36) for s in heatmap_data.index],
        text=heatmap_data.round(1).values,
        texttemplate="%{text}",
        colorscale="RdBu",
        reversescale=True,
        zmin=-1.8,
        zmax=1.8,
        colorbar=dict(title="Relativo"),
        hovertemplate="%{y}<br>%{x}: %{text}<br>Score relativo: %{z:.2f}<extra></extra>",
    ),
    row=2, col=1,
)

fig.update_layout(
    title="Visualizacion de clusters KMeans K=6",
    height=920,
    width=1350,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
    margin=dict(l=70, r=40, t=95, b=150),
)
fig.update_xaxes(title_text="Componente principal 1", row=1, col=1)
fig.update_yaxes(title_text="Componente principal 2", row=1, col=1)
fig.update_xaxes(title_text="Clientes", row=1, col=2)
fig.update_yaxes(automargin=True)

cluster_visualization_path = MODEL_DIR / "cluster_visualization.html"
fig.write_html(cluster_visualization_path, include_plotlyjs="cdn")
fig.show()

print("Visualizacion interactiva guardada en:", cluster_visualization_path)

Visualizacion interactiva guardada en: C:\Users\carol\OneDrive\Imágenes\Escritorio\Datathon\data_model_outputs\cluster_visualization.html


## 6. Anomaly Detection

Objetivo: detectar clientes con comportamiento fuera de patron antes de que reclamen.

Se usa `IsolationForest` sobre variables de riesgo y actividad. La salida se combina con reglas bancarias:

- muchas transacciones atipicas;
- disputas;
- actividad internacional;
- cargos no reconocidos;
- rechazos o reintentos.

In [8]:
anomaly_features = [
    "monto_total", "monto_max", "num_no_procesadas", "num_disputas", "num_reintentos",
    "num_transacciones_internacionales", "num_patrones_atipicos", "ratio_atipicas", "ratio_internacional",
    "intent_cargo_no_reconocido", "riesgo_fraude_score", "riesgo_friccion_score",
]
anomaly_features = [c for c in anomaly_features if c in customer_360.columns]
X_anomaly = customer_360[anomaly_features].apply(pd.to_numeric, errors="coerce").fillna(0)
X_anomaly_scaled = StandardScaler().fit_transform(X_anomaly)

iso = IsolationForest(n_estimators=250, contamination=0.05, random_state=42)
anomaly_pred = iso.fit_predict(X_anomaly_scaled)
anomaly_score_raw = -iso.score_samples(X_anomaly_scaled)

customer_360["anomaly_flag"] = (anomaly_pred == -1).astype(int)
customer_360["anomaly_score"] = minmax_score(pd.Series(anomaly_score_raw, index=customer_360.index))

anomaly_summary = customer_360.groupby("anomaly_flag").agg(
    clientes=("user_id", "count"),
    fraude_promedio=("riesgo_fraude_score", "mean"),
    disputas_promedio=("num_disputas", "mean"),
    atipicas_promedio=("num_patrones_atipicos", "mean"),
    internacionales_promedio=("num_transacciones_internacionales", "mean"),
).round(2).reset_index()

anomaly_summary

,anomaly_flag,clientes,fraude_promedio,disputas_promedio,atipicas_promedio,internacionales_promedio
0,0,14273,8.11,1.24,0.31,2.63
1,1,752,28.17,1.74,49.25,3.35


## 7. Modelos de propension y churn/friction

En esta fase se consolidan scores existentes en variables listas para decision.

Nota de produccion:

- En un entorno real, estos scores pueden entrenarse con labels de aceptacion, conversion, churn y CSAT.
- En este hackathon, se usan scores explicables derivados de comportamiento para demostrar la logica de decision.

In [9]:
customer_360["propension_ahorro"] = customer_360["capacidad_ahorro_score"]
customer_360["propension_inversion"] = customer_360["propension_inversion_score"]
customer_360["propension_credito"] = customer_360["propension_credito_score"]
customer_360["propension_seguro"] = customer_360["propension_seguro_score"]
customer_360["propension_hey_pro"] = (
    0.35 * minmax_score(customer_360["cashback_total"]) +
    0.25 * minmax_score(customer_360["monto_total"]) +
    0.20 * minmax_score(1 - customer_360["es_hey_pro"].astype(int)) +
    0.20 * minmax_score(customer_360["satisfaccion_1_10"])
).round(2).clip(0, 100)
customer_360["propension_alerta_gasto"] = (
    0.45 * customer_360["estres_financiero_score"] +
    0.25 * minmax_score(customer_360["gasto_delivery"] + customer_360["gasto_restaurante"] + customer_360["gasto_servicios_digitales"]) +
    0.30 * minmax_score(customer_360["rechazos_saldo_insuficiente"])
).round(2).clip(0, 100)
customer_360["propension_contacto_humano"] = (
    0.40 * customer_360["riesgo_friccion_score"] +
    0.25 * minmax_score(customer_360["intent_asesor_humano"]) +
    0.20 * minmax_score(customer_360["num_disputas"]) +
    0.15 * minmax_score(customer_360["intent_cancelacion"])
).round(2).clip(0, 100)
customer_360["churn_friction_risk_model"] = customer_360["churn_risk_score"]

propensity_cols = [
    "propension_ahorro", "propension_inversion", "propension_credito", "propension_seguro",
    "propension_hey_pro", "propension_alerta_gasto", "propension_contacto_humano", "churn_friction_risk_model",
]
customer_360[["user_id"] + propensity_cols].head()

,user_id,propension_ahorro,propension_inversion,propension_credito,propension_seguro,propension_hey_pro,propension_alerta_gasto,propension_contacto_humano,churn_friction_risk_model
0,USR-00001,33.59,64.75,28.68,52.02,22.45,18.25,1.72,4.44
1,USR-00002,20.62,55.28,35.94,16.21,18.38,13.63,11.98,10.67
2,USR-00003,17.98,52.98,19.17,48.13,18.09,24.91,14.5,14.39
3,USR-00004,53.94,48.32,50.37,39.29,49.59,7.48,14.14,11.28
4,USR-00005,24.47,50.09,29.48,21.24,15.22,9.17,10.72,15.91


## 8. Need Detection Engine

Esta capa junta los modelos y scores para inferir necesidades implicitas.

La salida principal es `necesidad_implicita`, por ejemplo:

- `proteccion_y_aclaracion`;
- `control_financiero`;
- `ahorro_inversion`;
- `credito_responsable`;
- `retencion_friccion`;
- `proteccion_seguro`;
- `engagement_digital`.

La logica prioriza necesidades de riesgo antes que oportunidades comerciales.

In [10]:
def detect_need(row):
    # 1. Riesgo de seguridad o aclaracion: prioridad maxima.
    if (
        row.get("riesgo_fraude_score", 0) >= 55
        or row.get("anomaly_flag", 0) == 1 and row.get("intent_cargo_no_reconocido", 0) > 0
        or row.get("num_disputas", 0) >= 4
    ):
        return "proteccion_y_aclaracion"

    # 2. Friccion o churn.
    if row.get("riesgo_friccion_score", 0) >= 55 or row.get("churn_risk_score", 0) >= 55 or row.get("intent_cancelacion", 0) > 0:
        return "retencion_friccion"

    # 3. Control financiero.
    if row.get("estres_financiero_score", 0) >= 45 or row.get("rechazos_saldo_insuficiente", 0) >= 2 or row.get("propension_alerta_gasto", 0) >= 55:
        return "control_financiero"

    # 4. Ahorro/inversion.
    if row.get("propension_inversion", 0) >= 55 and row.get("tiene_inversion", 0) == 0:
        return "ahorro_inversion"

    # 5. Credito responsable.
    if row.get("propension_credito", 0) >= 55 or row.get("intent_credito", 0) > 0:
        return "credito_responsable"

    # 6. Seguro/proteccion.
    if row.get("propension_seguro", 0) >= 55 and row.get("tiene_seguro", 0) == 0:
        return "proteccion_seguro"

    # 7. Engagement digital.
    if row.get("total_interacciones_havi", 0) >= 5 or row.get("dias_desde_ultimo_login", 0) >= 60:
        return "engagement_digital"

    return "educacion_financiera"


def need_risk_level(row):
    risk = max(
        row.get("riesgo_fraude_score", 0),
        row.get("riesgo_friccion_score", 0),
        row.get("estres_financiero_score", 0),
        row.get("churn_risk_score", 0),
        row.get("anomaly_score", 0),
    )
    if risk >= 70:
        return "alto"
    if risk >= 40:
        return "medio"
    return "bajo"

customer_360["necesidad_implicita"] = customer_360.apply(detect_need, axis=1)
customer_360["riesgo"] = customer_360.apply(need_risk_level, axis=1)

need_summary = customer_360.groupby(["necesidad_implicita", "riesgo"]).agg(
    clientes=("user_id", "count"),
    fraude_promedio=("riesgo_fraude_score", "mean"),
    friccion_promedio=("riesgo_friccion_score", "mean"),
    estres_promedio=("estres_financiero_score", "mean"),
    inversion_promedio=("propension_inversion", "mean"),
).round(2).reset_index().sort_values("clientes", ascending=False)
need_summary

C:\Users\carol\AppData\Local\Temp\ipykernel_19528\3925632834.py:52: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



,necesidad_implicita,riesgo,clientes,fraude_promedio,friccion_promedio,estres_promedio,inversion_promedio
1,ahorro_inversion,bajo,3571,8.63,9.3,12.62,62.10
10,educacion_financiera,bajo,2475,7.64,12.97,13.04,43.66
7,credito_responsable,bajo,2466,6.76,11.76,10.61,43.10
16,proteccion_seguro,bajo,2054,6.60,10.99,9.94,41.96
22,retencion_friccion,bajo,964,7.57,14.68,12.57,48.88
13,engagement_digital,bajo,904,4.08,12.37,6.43,42.66
19,proteccion_y_aclaracion,bajo,532,18.08,17.58,11.71,53.56
20,proteccion_y_aclaracion,medio,333,20.53,20.82,14.9,49.06
2,ahorro_inversion,medio,250,21.49,10.97,14.61,62.17
11,educacion_financiera,medio,245,15.19,16.62,18.13,43.00


## 9. Next Best Action Engine

El motor de accion combina:

- necesidad implicita;
- riesgo;
- propension;
- restricciones de negocio;
- canal recomendado;
- mensaje personalizado.

No es solo ML: es una capa de decision explicable y controlada.

In [11]:
# Catalogo completo de acciones de negocio.
# La decision final combina necesidad detectada, propensiones, riesgo y reglas de seguridad.
action_catalog = {
    "activar_alerta_gasto": {
        "canal": "Havi + push",
        "kpi": "reduccion de cargos rechazados y mayor control financiero",
        "mensaje": "Detecte aumentos de gasto o rechazos recientes. Puedo activar una alerta semanal para ayudarte a mantenerte dentro de tu presupuesto.",
    },
    "crear_meta_ahorro": {
        "canal": "App + Havi",
        "kpi": "aumento de ahorro recurrente y engagement",
        "mensaje": "Veo una oportunidad para separar dinero de forma automatica. Puedo ayudarte a crear una meta de ahorro semanal ajustada a tu flujo.",
    },
    "ofrecer_inversion": {
        "canal": "App + Havi",
        "kpi": "conversion a inversion y aumento de saldo invertido",
        "mensaje": "Tienes capacidad para hacer crecer tu dinero. Puedo mostrarte una opcion de inversion Hey acorde a tu perfil.",
    },
    "ofrecer_credito": {
        "canal": "Havi + app",
        "kpi": "conversion responsable a credito",
        "mensaje": "Puedo mostrarte una opcion de credito con mensualidades estimadas segun tu perfil y capacidad financiera.",
    },
    "bloquear_tarjeta": {
        "canal": "Havi + push urgente",
        "kpi": "reduccion de fraude y tiempo de reaccion ante cargos atipicos",
        "mensaje": "Detecte actividad inusual. Por seguridad, puedo ayudarte a bloquear temporalmente tu tarjeta mientras revisamos los movimientos.",
    },
    "iniciar_aclaracion": {
        "canal": "Havi + seguimiento en app",
        "kpi": "reduccion de tickets repetidos y mejora de satisfaccion",
        "mensaje": "Detecte senales de cargos no reconocidos o disputas. Puedo iniciar una aclaracion asistida y dar seguimiento desde la app.",
    },
    "derivar_asesor": {
        "canal": "Havi + asesor humano",
        "kpi": "reduccion de churn y mejora de CSAT",
        "mensaje": "Note friccion reciente en tu experiencia. Puedo conectarte con un asesor para resolverlo con prioridad.",
    },
    "explicar_movimiento": {
        "canal": "Havi",
        "kpi": "reduccion de consultas repetidas y mayor claridad transaccional",
        "mensaje": "Puedo explicarte tus movimientos recientes, identificar categorias y ayudarte a reconocer cargos recurrentes.",
    },
    "recordar_pago": {
        "canal": "Push + app",
        "kpi": "reduccion de pagos tardios y rechazos por saldo insuficiente",
        "mensaje": "Detecte posibles pagos o mensualidades cercanas. Puedo enviarte recordatorios para evitar atrasos o rechazos.",
    },
    "ofrecer_seguro": {
        "canal": "App + push",
        "kpi": "conversion a seguro y mayor proteccion del cliente",
        "mensaje": "Podrias proteger mejor tus compras o tu estabilidad financiera. Puedo mostrarte opciones de seguro acordes a tu perfil.",
    },
    "enviar_promocion": {
        "canal": "App + push",
        "kpi": "aumento de CTR y uso de beneficios personalizados",
        "mensaje": "Encontre una promocion relevante segun tus categorias de gasto frecuentes. Puedo mostrartela en la app.",
    },
}

# Escalas globales para evitar scoring con min-max de una sola fila.
score_helpers = {
    "disputas_scaled": minmax_score(customer_360["num_disputas"]),
    "rechazos_saldo_scaled": minmax_score(customer_360["rechazos_saldo_insuficiente"]),
    "score_buro_scaled": minmax_score(customer_360["score_buro"]),
    "intent_credito_scaled": minmax_score(customer_360["intent_credito"]),
    "edad_scaled": minmax_score(customer_360["edad"]),
    "interacciones_havi_scaled": minmax_score(customer_360["total_interacciones_havi"]),
    "dias_login_scaled": minmax_score(customer_360["dias_desde_ultimo_login"]),
    "cashback_scaled": minmax_score(customer_360["cashback_total"]),
    "monto_total_scaled": minmax_score(customer_360["monto_total"]),
    "mensualidad_scaled": minmax_score(customer_360["mensualidad_total_creditos"]),
}
for col, values in score_helpers.items():
    customer_360[col] = values


def score_action(row, action):
    if action == "bloquear_tarjeta":
        return 0.55 * row["riesgo_fraude_score"] + 0.30 * row["anomaly_score"] + 0.15 * row["urgencia_score"]
    if action == "iniciar_aclaracion":
        return 0.45 * row["riesgo_fraude_score"] + 0.25 * row["disputas_scaled"] + 0.20 * row["intent_cargo_no_reconocido"] * 100 + 0.10 * row["frustracion_score"]
    if action == "derivar_asesor":
        return 0.40 * row["riesgo_friccion_score"] + 0.30 * row["churn_risk_score"] + 0.20 * row["propension_contacto_humano"] + 0.10 * row["intent_asesor_humano"] * 100
    if action == "activar_alerta_gasto":
        return 0.45 * row["estres_financiero_score"] + 0.35 * row["propension_alerta_gasto"] + 0.20 * row["rechazos_saldo_scaled"]
    if action == "crear_meta_ahorro":
        return 0.55 * row["propension_ahorro"] + 0.25 * row["capacidad_ahorro_score"] + 0.20 * row["satisfaccion_1_10"] * 10
    if action == "ofrecer_inversion":
        return 0.60 * row["propension_inversion"] + 0.25 * row["capacidad_ahorro_score"] + 0.15 * row["monto_total_scaled"]
    if action == "ofrecer_credito":
        return 0.50 * row["propension_credito"] + 0.30 * row["score_buro_scaled"] + 0.20 * row["intent_credito_scaled"]
    if action == "ofrecer_seguro":
        return 0.60 * row["propension_seguro"] + 0.25 * row["edad_scaled"] + 0.15 * row["monto_total_scaled"]
    if action == "explicar_movimiento":
        return 0.45 * min(row.get("intent_consulta_movimiento", 0), 1) * 100 + 0.30 * row["interacciones_havi_scaled"] + 0.25 * row["frustracion_score"]
    if action == "recordar_pago":
        return 0.45 * row["mensualidad_scaled"] + 0.35 * row["rechazos_saldo_scaled"] + 0.20 * row["estres_financiero_score"]
    if action == "enviar_promocion":
        return 0.45 * row["cashback_scaled"] + 0.25 * row["monto_total_scaled"] + 0.20 * min(row.get("intent_promocion", 0), 1) * 100 + 0.10 * row["propension_hey_pro"]
    return 0


def candidate_actions(row):
    candidates = []

    # Seguridad tiene prioridad por riesgo operativo y regulatorio.
    if row["riesgo_fraude_score"] >= 70 or (row["anomaly_flag"] == 1 and row["anomaly_score"] >= 70):
        candidates.append("bloquear_tarjeta")
    if row["necesidad_implicita"] == "proteccion_y_aclaracion" or row["intent_cargo_no_reconocido"] > 0 or row["num_disputas"] > 0:
        candidates.append("iniciar_aclaracion")

    # Friccion, churn y soporte humano.
    if row["necesidad_implicita"] == "retencion_friccion" or row["propension_contacto_humano"] >= 55 or row["intent_asesor_humano"] > 0:
        candidates.append("derivar_asesor")

    # Salud financiera y ahorro.
    if row["necesidad_implicita"] == "control_financiero" or row["propension_alerta_gasto"] >= 55:
        candidates.append("activar_alerta_gasto")
    if row["propension_ahorro"] >= 50 and row.get("tiene_inversion", 0) == 0:
        candidates.append("crear_meta_ahorro")
    if row["necesidad_implicita"] == "ahorro_inversion" or row["propension_inversion"] >= 55:
        candidates.append("ofrecer_inversion")

    # Productos y beneficios.
    if row["necesidad_implicita"] == "credito_responsable" or row["propension_credito"] >= 55 or row["intent_credito"] > 0:
        candidates.append("ofrecer_credito")
    if row["necesidad_implicita"] == "proteccion_seguro" or row["propension_seguro"] >= 55:
        candidates.append("ofrecer_seguro")
    if row["propension_hey_pro"] >= 55 or row.get("intent_promocion", 0) > 0 or row["cashback_total"] > 0:
        candidates.append("enviar_promocion")

    # Operacion y educacion financiera.
    if row.get("intent_consulta_movimiento", 0) > 0 or row["necesidad_implicita"] == "educacion_financiera":
        candidates.append("explicar_movimiento")
    if row["mensualidad_total_creditos"] > 0 or row["rechazos_saldo_insuficiente"] > 0:
        candidates.append("recordar_pago")

    # Fallback: siempre debe existir una accion explicable.
    if not candidates:
        candidates.append("explicar_movimiento")

    return list(dict.fromkeys(candidates))


def choose_next_best_action(row):
    candidates = candidate_actions(row)
    scored = [(action, score_action(row, action)) for action in candidates]
    return max(scored, key=lambda item: item[1])

nba_pairs = customer_360.apply(choose_next_best_action, axis=1)
customer_360["accion_recomendada"] = nba_pairs.map(lambda x: x[0])
customer_360["action_score"] = nba_pairs.map(lambda x: x[1]).round(2).clip(0, 100)
customer_360["canal"] = customer_360["accion_recomendada"].map(lambda a: action_catalog[a]["canal"])
customer_360["kpi_esperado"] = customer_360["accion_recomendada"].map(lambda a: action_catalog[a]["kpi"])
customer_360["mensaje"] = customer_360["accion_recomendada"].map(lambda a: action_catalog[a]["mensaje"])

next_best_actions = customer_360[[
    "user_id", "segmento", "necesidad_implicita", "riesgo", "accion_recomendada", "action_score", "canal", "mensaje", "kpi_esperado"
]].sort_values("action_score", ascending=False)

next_best_actions.head(15)



C:\Users\carol\AppData\Local\Temp\ipykernel_19528\2973681594.py:75: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\carol\AppData\Local\Temp\ipykernel_19528\2973681594.py:75: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\carol\AppData\Local\Temp\ipykernel_19528\2973681594.py:75: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

C:\Users\carol\AppData\Local\Temp\ipykernel_19528\2973681594.py:152: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\carol\AppData\Local\Temp\ipykernel_19528\2973681594.py:153: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\carol\AppData\Local\Temp\ipykernel_19528\2973681594.py:154: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragm

,user_id,segmento,necesidad_implicita,riesgo,accion_recomendada,action_score,canal,mensaje,kpi_esperado
369,USR-00370,Candidatos a credito,proteccion_y_aclaracion,medio,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
4062,USR-04063,Alertas criticas de fraude,proteccion_y_aclaracion,alto,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
3747,USR-03748,Clientes con estres financiero,proteccion_y_aclaracion,medio,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
7494,USR-07495,Candidatos a credito,proteccion_y_aclaracion,medio,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
14466,USR-14467,Clientes de bajo engagement,engagement_digital,medio,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
3145,USR-03146,Clientes con estres financiero,proteccion_y_aclaracion,medio,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
5412,USR-05413,Candidatos a credito,proteccion_y_aclaracion,alto,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
10398,USR-10399,Inversionistas de alto valor en vigilancia,proteccion_y_aclaracion,alto,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
1308,USR-01309,Clientes de bajo engagement,engagement_digital,medio,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...
8149,USR-08150,Candidatos a credito,proteccion_y_aclaracion,medio,iniciar_aclaracion,100.0,Havi + seguimiento en app,Detecte senales de cargos no reconocidos o dis...,reduccion de tickets repetidos y mejora de sat...


## 10. Salida tipo JSON por cliente

Esta salida es la que se puede mostrar en dashboard, API o demo de Havi.

In [12]:
sample_user = next_best_actions.iloc[0].to_dict()
sample_user

{'user_id': 'USR-00370',
 'segmento': 'Candidatos a credito',
 'necesidad_implicita': 'proteccion_y_aclaracion',
 'riesgo': 'medio',
 'accion_recomendada': 'iniciar_aclaracion',
 'action_score': 100.0,
 'canal': 'Havi + seguimiento en app',
 'mensaje': 'Detecte senales de cargos no reconocidos o disputas. Puedo iniciar una aclaracion asistida y dar seguimiento desde la app.',
 'kpi_esperado': 'reduccion de tickets repetidos y mejora de satisfaccion'}

## 11. Evaluacion de modelos y calidad de decision

Metrica y evidencia para pitch:

- NLP: distribucion de intenciones y volumen de casos sensibles.
- Clustering: silhouette, Davies-Bouldin, tamanos de cluster e interpretabilidad.
- Anomaly Detection: tasa de anomalias y cruce con disputas/patrones atipicos.
- Recomendaciones: distribucion de acciones y priorizacion por `action_score`.
- Negocio: KPIs esperados por accion.

In [13]:
nlp_eval = customer_360["intencion_dominante"].value_counts().reset_index()
nlp_eval.columns = ["intencion_dominante", "clientes"]

action_eval = customer_360.groupby(["necesidad_implicita", "accion_recomendada", "kpi_esperado"]).agg(
    clientes=("user_id", "count"),
    action_score_promedio=("action_score", "mean"),
    riesgo_alto=("riesgo", lambda s: (s == "alto").sum()),
).round(2).reset_index().sort_values("clientes", ascending=False)

model_evaluation_summary = {
    "best_cluster_metrics": cluster_metrics_df.head(1).to_dict("records")[0],
    "selected_k": K_SEGMENTS,
    "anomaly_rate": float(round(customer_360["anomaly_flag"].mean(), 4)),
    "total_customers": int(len(customer_360)),
    "unique_segments": int(customer_360["segmento"].nunique()),
    "unique_needs": int(customer_360["necesidad_implicita"].nunique()),
    "unique_actions": int(customer_360["accion_recomendada"].nunique()),
}

print(model_evaluation_summary)
display(nlp_eval)
display(action_eval)

{'best_cluster_metrics': {'k': 6, 'silhouette_score': 0.12237626268716378, 'davies_bouldin_score': 2.1751550980710213}, 'selected_k': 6, 'anomaly_rate': 0.05, 'total_customers': 15025, 'unique_segments': 6, 'unique_needs': 8, 'unique_actions': 11}


,intencion_dominante,clientes
0,credito,3374
1,bloqueo_tarjeta,3075
2,consulta_movimiento,2966
3,sin_intencion_detectada,2914
4,cargo_no_reconocido,720
5,soporte_app,681
6,asesor_humano,640
7,cancelacion,352
8,promocion,173
9,cashback,130


,necesidad_implicita,accion_recomendada,kpi_esperado,clientes,action_score_promedio,riesgo_alto
46,proteccion_seguro,ofrecer_seguro,conversion a seguro y mayor proteccion del cli...,2206,57.99,23
6,ahorro_inversion,ofrecer_inversion,conversion a inversion y aumento de saldo inve...,1830,49.50,21
23,credito_responsable,ofrecer_credito,conversion responsable a credito,1360,40.20,11
7,ahorro_inversion,ofrecer_seguro,conversion a seguro y mayor proteccion del cli...,1304,62.28,15
25,credito_responsable,ofrecer_seguro,conversion a seguro y mayor proteccion del cli...,972,59.06,9
30,educacion_financiera,explicar_movimiento,reduccion de consultas repetidas y mayor clari...,898,37.58,6
31,educacion_financiera,iniciar_aclaracion,reduccion de tickets repetidos y mejora de sat...,673,13.81,0
38,engagement_digital,explicar_movimiento,reduccion de consultas repetidas y mayor clari...,624,29.89,1
29,educacion_financiera,enviar_promocion,aumento de CTR y uso de beneficios personalizados,463,14.84,0
34,educacion_financiera,recordar_pago,reduccion de pagos tardios y rechazos por sald...,463,17.12,0


## 12. Exportar outputs de Fase 3

Se materializan los resultados para dashboard, app, storytelling o API.

In [14]:
customer_360.to_pickle(MODEL_DIR / "customer_360_scored.pkl")
segment_summary.to_pickle(MODEL_DIR / "segment_summary.pkl")
need_summary.to_pickle(MODEL_DIR / "need_summary.pkl")
next_best_actions.to_pickle(MODEL_DIR / "next_best_actions.pkl")
action_eval.to_pickle(MODEL_DIR / "action_eval.pkl")

print("Outputs Fase 3 guardados en:", MODEL_DIR)
for file in MODEL_DIR.glob("*.pkl"):
    print(file.name, file.stat().st_size)

Outputs Fase 3 guardados en: C:\Users\carol\OneDrive\Imágenes\Escritorio\Datathon\data_model_outputs
action_eval.pkl 4486
customer_360_scored.pkl 17188305
need_summary.pkl 2766
next_best_actions.pkl 950538
segment_summary.pkl 2005


## 13. Resultado de Fase 3

La capa inteligente queda lista:

```text
Customer 360 -> Segmentos -> Anomalias -> Necesidades implicitas -> Next Best Action
```

Outputs clave:

- `customer_360_scored.pkl`: base completa con modelos y acciones.
- `segment_summary.pkl`: resumen de clusters interpretables.
- `need_summary.pkl`: necesidades detectadas por volumen y riesgo.
- `next_best_actions.pkl`: tabla priorizada de acciones por cliente.
- `action_eval.pkl`: resumen de acciones y KPIs esperados.

Siguiente fase recomendada: dashboard/demo interactiva de Hey Sense.